# 03 — Integração dos Dados e Construção da Camada Gold

Este notebook integra as bases tratadas na camada Silver e constrói a camada Gold utilizada nas análises do MVP. O ano de 2013 é adotado como recorte analítico por apresentar elevada cobertura simultânea entre as fontes, especialmente para os indicadores de desenvolvimento humano.

Antes da integração, são aplicadas as regras de qualidade definidas na etapa anterior. Em seguida, as bases WHO e IDH são conciliadas pela chave composta país-ano e organizadas em um modelo dimensional em esquema estrela, composto pelas dimensões de país e tempo e por uma tabela fato contendo os indicadores de saúde e desenvolvimento.

1. Carregar as tabelas Silver

In [0]:
# Importar as funções utilizadas na integração e modelagem 
from pyspark.sql import functions as F 
from pyspark.sql.window import Window 

# Carregar as tabelas tratadas da camada Silver 
df_who = spark.table("workspace.silver.who_life_expectancy") 

df_idh = spark.table("workspace.silver.idh") 

print(f"WHO Silver: {df_who.count()} registros") 
print(f"IDH Silver: {df_idh.count()} registros")

WHO Silver: 2938 registros
IDH Silver: 5336 registros


2. Recorte de 2013 e aplicação das regras de qualidade

In [0]:
# Aplicar o recorte temporal de 2013 e mantém apenas registros válidos da WHO 
df_who_2013 = ( df_who .filter( (F.col("ano") == 2013) & (F.col("flag_registro_invalido") == 0) ) ) 

# Aplicar o recorte de 2013 e remove da integração as chaves ambíguas da base IDH 
df_idh_2013 = (df_idh .filter( 
                (F.col("ano") == 2013) & (F.col("flag_chave_duplicada") == 0) ) 
               .withColumn( "indice_educacao", 
                    F.when( 
                     F.col("flag_indice_educacao_invalido") == 1, F.lit(None).cast("double") )
                    .otherwise(F.col("indice_educacao")) ) ) 

print(f"WHO após recorte: {df_who_2013.count()} registros") 
print(f"IDH após recorte e qualidade: {df_idh_2013.count()} registros")

WHO após recorte: 193 registros
IDH após recorte e qualidade: 180 registros


3. Avaliação da cobertura entre as fontes

In [0]:
# Verificar a quantidade de países disponíveis antes da integração 
print( "Países WHO:", df_who_2013.select("pais").distinct().count() ) 
print( "Países IDH:", df_idh_2013.select("pais").distinct().count() ) 

# Identificar países da WHO sem correspondência na base IDH 
paises_sem_correspondencia = ( df_who_2013 .select("pais") .distinct() .join( df_idh_2013.select("pais").distinct(), on="pais", how="left_anti" ) ) 

print( "Países WHO sem correspondência na IDH:", paises_sem_correspondencia.count() ) 
display(paises_sem_correspondencia.orderBy("pais"))

Países WHO: 193
Países IDH: 180
Países WHO sem correspondência na IDH: 13


pais
China
Congo
Cook Islands
Democratic People's Republic of Korea
Democratic Republic of the Congo
Lao People's Democratic Republic
Monaco
Nauru
Niue
San Marino


4. Integrar WHO e IDH

In [0]:
# Integrar as duas fontes utilizando país e ano como chave composta 
df_integrado = ( df_who_2013 .join( df_idh_2013.select( "pais", "ano", "anos_escolaridade_media", "indice_educacao", "idh" ), on=["pais", "ano"], how="inner" ) ) 

# Verificar a dimensão do conjunto integrado 
print("Registros integrados:", df_integrado.count()) 
print( "Países integrados:", df_integrado.select("pais").distinct().count() ) 

# Confirmar que o JOIN não criou duplicidades na chave país-ano 
duplicatas_integracao = ( df_integrado .groupBy("pais", "ano") .count() .filter(F.col("count") > 1) .count() ) 
print("Chaves duplicadas após o JOIN:", duplicatas_integracao) 
display(df_integrado.limit(10))

Registros integrados: 180
Países integrados: 180
Chaves duplicadas após o JOIN: 0


pais,ano,status,exp_vida,mortalidade_adulta,mortalidade_infantil,mortalidade_hiv,consumo_alcool,imc,pib_dolar,flag_registro_invalido,anos_escolaridade_media,indice_educacao,idh
Afghanistan,2013,Developing,59.9,268.0,66.0,0.1,0.01,18.1,631.744976,0,10.2,0.4372222222222222,0.484962004929948
Albania,2013,Developing,77.2,84.0,0.0,0.1,4.76,56.5,4414.72314,0,14.8,0.7627777777777778,0.781171149321141
Algeria,2013,Developing,75.3,112.0,21.0,0.1,0.53,57.2,5471.866766,0,14.3,0.6933333333333334,0.745817134419061
Angola,2013,Developing,51.1,355.0,69.0,2.3,8.1,22.1,484.616884,0,10.3,0.4794444444444445,0.547266233129838
Antigua and Barbuda,2013,Developing,76.1,133.0,0.0,0.2,8.58,46.4,12224.86416,0,13.0,0.6888888888888889,0.767347961741302
Argentina,2013,Developing,76.0,119.0,8.0,0.1,8.28,61.6,12976.63642,0,17.1,0.8422222222222223,0.824245166877387
Armenia,2013,Developing,74.4,123.0,1.0,0.1,3.79,53.3,3843.591213,0,13.0,0.75,0.7434815305781
Australia,2013,Developed,82.5,61.0,1.0,0.1,9.87,65.5,67792.3386,0,20.4,null,0.926368881043257
Austria,2013,Developed,81.1,68.0,0.0,0.1,11.82,56.6,554.71532,0,15.8,0.8572222222222223,0.896416350346645
Azerbaijan,2013,Developing,72.2,121.0,5.0,0.1,2.14,5.6,7875.756953,0,12.2,0.7066666666666667,0.740645466085216


5. Construção das dimensões

In [0]:
# Cria a dimensão País com uma chave numérica para cada país 
janela_pais = Window.orderBy("pais") 
dim_pais = ( df_integrado .select("pais", "status") .distinct() 
            .withColumn( "id_pais", F.row_number().over(janela_pais)) 
            .select( "id_pais", "pais", "status" )) 

# Cria a dimensão Tempo correspondente ao recorte do MVP 
dim_tempo = ( df_integrado .select("ano") .distinct() 
             .withColumn("id_tempo", F.col("ano")) 
             .select( "id_tempo", "ano" ) ) 

display(dim_pais.limit(10)) 
display(dim_tempo)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_pais,pais,status
1,Afghanistan,Developing
2,Albania,Developing
3,Algeria,Developing
4,Angola,Developing
5,Antigua and Barbuda,Developing
6,Argentina,Developing
7,Armenia,Developing
8,Australia,Developed
9,Austria,Developed
10,Azerbaijan,Developing


id_tempo,ano
2013,2013


6. Construção da tabela fato

In [0]:
# Associa os indicadores integrados às chaves das dimensões País e Tempo 
fato_saude_desenvolvimento = ( df_integrado .join( dim_pais, on=["pais", "status"], how="inner")
                               .join( dim_tempo, on="ano", how="inner" )
                               .select("id_pais", "id_tempo", "exp_vida", "mortalidade_adulta", "mortalidade_infantil", "mortalidade_hiv", "consumo_alcool", "imc", "pib_dolar", "anos_escolaridade_media", "indice_educacao", "idh")) 

display(fato_saude_desenvolvimento.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_pais,id_tempo,exp_vida,mortalidade_adulta,mortalidade_infantil,mortalidade_hiv,consumo_alcool,imc,pib_dolar,anos_escolaridade_media,indice_educacao,idh
1,2013,59.9,268.0,66.0,0.1,0.01,18.1,631.744976,10.2,0.4372222222222222,0.484962004929948
2,2013,77.2,84.0,0.0,0.1,4.76,56.5,4414.72314,14.8,0.7627777777777778,0.781171149321141
3,2013,75.3,112.0,21.0,0.1,0.53,57.2,5471.866766,14.3,0.6933333333333334,0.745817134419061
4,2013,51.1,355.0,69.0,2.3,8.1,22.1,484.616884,10.3,0.4794444444444445,0.547266233129838
5,2013,76.1,133.0,0.0,0.2,8.58,46.4,12224.86416,13.0,0.6888888888888889,0.767347961741302
6,2013,76.0,119.0,8.0,0.1,8.28,61.6,12976.63642,17.1,0.8422222222222223,0.824245166877387
7,2013,74.4,123.0,1.0,0.1,3.79,53.3,3843.591213,13.0,0.75,0.7434815305781
8,2013,82.5,61.0,1.0,0.1,9.87,65.5,67792.3386,20.4,null,0.926368881043257
9,2013,81.1,68.0,0.0,0.1,11.82,56.6,554.71532,15.8,0.8572222222222223,0.896416350346645
10,2013,72.2,121.0,5.0,0.1,2.14,5.6,7875.756953,12.2,0.7066666666666667,0.740645466085216


7. Persistência da camada Gold

In [0]:
# Dimensões e a tabela fato em formato Delta na camada Gold
dim_pais.write.format("delta").mode("overwrite") .option("overwriteSchema", "true") .saveAsTable("workspace.gold.dim_pais")

dim_tempo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_tempo")

fato_saude_desenvolvimento.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.fato_saude_desenvolvimento")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


8. Validação da camada Gold

In [0]:
# Confirmar a criação das tabelas e verificar a quantidade final de registros 
spark.sql("SHOW TABLES IN workspace.gold").show()
 
print( "Países na dimensão:", spark.table("workspace.gold.dim_pais").count() ) 
print( "Registros na tabela fato:", spark.table("workspace.gold.fato_saude_desenvolvimento").count() )

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|    gold|            dim_pais|      false|
|    gold|           dim_tempo|      false|
|    gold|fato_saude_desenv...|      false|
+--------+--------------------+-----------+

Países na dimensão: 180
Registros na tabela fato: 180


As bases WHO e IDH foram integradas após a aplicação do recorte de 2013 e das regras de qualidade definidas na camada Silver. A correspondência entre as fontes foi avaliada antes do JOIN, e a unicidade da chave país-ano foi verificada após a integração.

O conjunto integrado foi organizado em um modelo estrela, composto pelas dimensões País e Tempo e pela tabela fato de indicadores de saúde e desenvolvimento. As estruturas resultantes foram persistidas em formato Delta na camada Gold e serão utilizadas como fonte para as análises do MVP.